# Notebook 04 — Closed-Loop Dual-Track Generation (With WQB)

**Goal**: Run the full dual-track orchestration loop:
```
[Explorer]  Idea → Synthesis → Validate → Novelty(strict)  ──┐
                                                               ├─► WQB Simulate → Qualify → Reflect
[Skeleton]  SkeletonRegistry → SkeletonAgent → Validate ──────┘         │
                                                                         ▼
                                                              SkeletonExtractor → SkeletonRegistry
```

**Three `track_mode` options:**
- `"explorer_only"` — only the free-exploration LLM track (original behavior)
- `"skeleton_only"` — only the controlled skeleton-variation track
- `"hybrid"` *(default)* — both tracks, budget split by TrackBandit (UCB)

**Requires**: Notebooks 01–03 completed. WQB credentials set in `.env`.

⚠️ **WQB API quota**: each expression costs one simulation. Start with `n_rounds=1`.

In [1]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from alpha_agent.config import settings
from alpha_agent.knowledge.vector_store import VectorStore
from alpha_agent.knowledge.alpha_memory import AlphaMemory
from alpha_agent.knowledge.skeleton_registry import SkeletonRegistry
from alpha_agent.data.operator_kb import OperatorKB
from alpha_agent.data.wqb_client import WQBClient
from alpha_agent.engine.orchestrator import Orchestrator
from alpha_agent.search.bandit import DirectionBandit
from alpha_agent.search.track_bandit import TrackBandit

store    = VectorStore()
memory   = AlphaMemory()
skeleton_db_path = settings.duckdb_path.with_name("skeleton_registry.db")
registry = SkeletonRegistry(db_path=skeleton_db_path)
kb       = OperatorKB()

print("AlphaMemory DB:", settings.duckdb_path)
print("SkeletonRegistry DB:", skeleton_db_path)
print("SkeletonRegistry status:", registry.stats())

mem_status = memory.status()
print("AlphaMemory status:", mem_status)
if mem_status.get("read_only"):
    raise RuntimeError(
        "AlphaMemory is read-only in this kernel (likely DB lock by another notebook/process). "
        "Please stop other kernels using alpha_memory.db and rerun this cell."
    )

/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


AlphaMemory DB: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/alpha_memory.db
SkeletonRegistry DB: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/skeleton_registry.db
SkeletonRegistry status: {'total': 0, 'active': 0, 'instances': 0}
AlphaMemory status: {'path': '/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/alpha_memory.db', 'read_only': False, 'ephemeral': False, 'lock_message': ''}


## 1. Configure the run

In [2]:
# ─── Configuration ────────────────────────────────────────────────────────
# DIRECTION  = "short-term price reversal with liquidity adjustment"
DIRECTION = "medium-horizon reversal with liquidity and volatility normalization"
DATASET    = "pv1"
UNIVERSE   = "TOP1000"
N_ROUNDS   = 50        # start small! increase after validating setup
IDEAS_PER_ROUND   = 8
VARIANTS_PER_IDEA = 4
EXPLORE_BIAS = 0.3   # 0=explore, 1=exploit
AUTO_SUBMIT  = False  # set True to auto-submit qualified alphas
DRY_RUN      = False  # True = skip WQB (for testing prompt quality)

# ─── Track mode ───────────────────────────────────────────────────────────
# "hybrid"        → TrackBandit splits budget between Explorer + Skeleton
# "explorer_only" → only free-exploration LLM (original behavior)
# "skeleton_only" → only SkeletonAgent (needs seeds in registry first)
TRACK_MODE = "hybrid"
# ──────────────────────────────────────────────────────────────────────────

## 2. (Optional) Use Bandit to auto-select direction

In [3]:
# Uncomment to let the bandit choose the direction automatically
# bandit = DirectionBandit(memory)
# DIRECTION = bandit.select()
# print(f"Bandit selected direction: {DIRECTION}")

print(f"Direction: {DIRECTION}")
print(f"Dataset:   {DATASET} / {UNIVERSE}")
print(f"Rounds:    {N_ROUNDS}")
print(f"Dry run:   {DRY_RUN}")

Direction: medium-horizon reversal with liquidity and volatility normalization
Dataset:   pv1 / TOP1000
Rounds:    50
Dry run:   False


## 3. Run the orchestration loop

In [4]:
async def run_pipeline():
    async with WQBClient() as client:
        orch = Orchestrator(
            client=client,
            vector_store=store,
            alpha_memory=memory,
            skeleton_registry=registry,
            operator_kb=kb,
            auto_submit=AUTO_SUBMIT,
        )
        reports = await orch.run(
            direction=DIRECTION,
            dataset=DATASET,
            universe=UNIVERSE,
            n_rounds=N_ROUNDS,
            ideas_per_round=IDEAS_PER_ROUND,
            variants_per_idea=VARIANTS_PER_IDEA,
            explore_exploit_bias=EXPLORE_BIAS,
            dry_run=DRY_RUN,
            track_mode=TRACK_MODE,
        )
    return reports

reports = asyncio.run(run_pipeline())

──────────────── Round 1/50 — medium-horizon reversal with liquidity and volatility normalization ─────────────────

Allocation — explorer=32 | skeleton=0

[Explorer] Generating 8 ideas...

[Explorer] Synthesizing 4 variants/idea...

[Explorer] Generated 32 expressions

...

[Explorer] Generated 32 expressions

No expressions passed novelty filter.

                                       Round 50 Summary                                        
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                ┃                                                               Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ round                 │                                                                  50 │
│ direction             │ medium-horizon reversal with liquidity and volatility normalization │
│ ideas                 │                                                                  32 │
│ expressions_generated │                                                                  32 │
│ after_validation      │                                                                  32 │
│ after_novelty_filter  │                                                                   0 │
│ simulated             │                                                                   0 │
│ qualified             │                                                                   0 │
│ soft_qualified        │                                                                   0 │
│ skeletons_added       │                                                                   0 │
│ pass_rate             │                                                                 0.0 │
│ explorer_simulated    │                                                                   0 │
│ explorer_qualified    │                                                                   0 │
│ skeleton_simulated    │                                                                   0 │
│ skeleton_qualified    │                                                                   0 │
│ duration_s            │                                                          285.828481 │
└───────────────────────┴─────────────────────────────────────────────────────────────────────┘

──────────────────────────────────────────────── Session Complete ─────────────────────────────────────────────────

Total rounds: 50 | Simulated: 64 | Qualified: 0 | Pass rate: 0.0%

## 4. Review results

In [5]:
import pandas as pd

summary_rows = [r.to_dict() for r in reports]
df_summary = pd.DataFrame(summary_rows)
print("Round-by-round summary:")
display(df_summary)

Round-by-round summary:


,round,direction,ideas,expressions_generated,after_validation,after_novelty_filter,simulated,qualified,soft_qualified,skeletons_added,pass_rate,explorer_simulated,explorer_qualified,skeleton_simulated,skeleton_qualified,duration_s
0,1,medium-horizon reversal with liquidity and vol...,32,32,32,22,21,0,0,0,0.0,21,0,0,0,3640.740633
1,2,medium-horizon reversal with liquidity and vol...,32,32,32,6,3,0,0,0,0.0,3,0,0,0,789.935610
2,3,medium-horizon reversal with liquidity and vol...,32,32,32,2,2,0,0,0,0.0,2,0,0,0,708.227617
3,4,medium-horizon reversal with liquidity and vol...,30,30,30,0,0,0,0,0,0.0,0,0,0,0,324.262349
4,5,medium-horizon reversal with liquidity and vol...,32,32,32,4,4,0,0,0,0.0,4,0,0,0,870.271694
5,6,medium-horizon reversal with liquidity and vol...,32,32,32,5,2,0,0,0,0.0,2,0,0,0,804.020176
6,7,medium-horizon reversal with liquidity and vol...,32,32,32,1,1,0,0,0,0.0,1,0,0,0,540.225610
7,8,medium-horizon reversal with liquidity and vol...,32,32,32,0,0,0,0,0,0.0,0,0,0,0,305.583408
8,9,medium-horizon reversal with liquidity and vol...,32,32,32,0,0,0,0,0,0.0,0,0,0,0,348.484541
9,10,medium-horizon reversal with liquidity and vol...,32,32,32,0,0,0,0,0,0.0,0,0,0,0,394.115372


In [6]:
# Check alpha memory status
stats = memory.stats()
print(f"Alpha Memory after run:")
print(f"  Total:     {stats['total']}")
print(f"  Qualified: {stats['qualified']}")
print(f"  Pass rate: {stats['pass_rate']:.1%}")

Alpha Memory after run:
  Total:     80
  Qualified: 0
  Pass rate: 0.0%


In [7]:
# View top qualified alphas
top = memory.top_by_metric('sharpe', k=5, qualified_only=True)
print("Top 5 qualified alphas by Sharpe:")
for a in top:
    m = a['metrics']
    print(f"  sharpe={m.get('sharpe',0):.3f} fitness={m.get('fitness',0):.3f}")
    print(f"  {a['expression'][:90]}")
    print()

Top 5 qualified alphas by Sharpe:


In [8]:
# View recent reflections (learning)
recent = memory.recent(n=10)
failed = [a for a in recent if not a['qualified'] and a['reflection']]

print("Recent failure reflections:")
for a in failed[:3]:
    print(f"\n  Expression: {a['expression'][:60]}")
    print(f"  Reflection: {str(a['reflection'])[:300]}")

Recent failure reflections:

  Expression: if_else(ts_regression(volume, ts_step(1), 30, rettype=1) < 0
  Reflection: ```json
{
  "root_cause": "The alpha suffers from near-zero predictive power (IC = 0) due to ineffective signal construction, coupled with extreme concentration from a sparse binary condition. The 30-day reversal signal with volume confirmation fails to generate meaningful forecasts, likely because 

  Expression: if_else(ts_sum(returns, 20) < 0, -1 * zscore(ts_zscore(volum
  Reflection: {
  "root_cause": "The alpha expression fails due to a lack of predictive power and high turnover, primarily caused by a binary if_else condition that only activates for past losers, combined with an overcomplicated and ineffective volume signal. The near-zero IC mean (-0.000) indicates no correlati

  Expression: if_else(ts_sum(returns, 40) < 0, -1 * zscore(ts_delta(volume
  Reflection: {
  "root_cause": "The alpha lacks predictive power due to noisy and non-predictive signals, primari

## 5. Skeleton Registry — Results

In [9]:
# Skeleton registry stats after the run
reg_stats = registry.stats()
print("Skeleton Registry Stats:")
for k, v in reg_stats.items():
    print(f"  {k}: {v}")

# Show all active skeletons
skeletons = registry.all()
print(f"\nActive skeletons: {len(skeletons)}")
for sk in skeletons[:10]:
    print(f"\n  [{sk.skeleton_id[:8]}] {sk.template_str[:80]}")
    print(f"    operators: {sk.operators_used}")
    print(f"    field_slots: {[s['name'] for s in sk.field_slots]}")
    print(f"    attempts={sk.attempt_count}  qualified={sk.success_count}  rate={sk.success_rate:.0%}")

Skeleton Registry Stats:
  total: 0
  active: 0
  instances: 0

Active skeletons: 0


In [10]:
# TrackBandit stats
from alpha_agent.search.track_bandit import TrackBandit
bandit = TrackBandit(skeleton_registry=registry)
# Show allocation preview for different budgets
for budget in [5, 10, 20]:
    print(bandit.allocation_preview(budget))

# Bandit stats from reports
print("\nPer-round track performance:")
import pandas as pd
track_rows = []
for r in reports:
    track_rows.append({
        "round": r.round_num,
        "explorer_sim": r.explorer_simulated,
        "explorer_qual": r.explorer_qualified,
        "skeleton_sim": r.skeleton_simulated,
        "skeleton_qual": r.skeleton_qualified,
        "skeletons_added": r.skeletons_added,
    })
df_tracks = pd.DataFrame(track_rows)
display(df_tracks)

Budget=5: explorer=5 (100%) | skeleton=0 (0%)
Budget=10: explorer=10 (100%) | skeleton=0 (0%)
Budget=20: explorer=20 (100%) | skeleton=0 (0%)

Per-round track performance:


,round,explorer_sim,explorer_qual,skeleton_sim,skeleton_qual,skeletons_added
0,1,21,0,0,0,0
1,2,3,0,0,0,0
2,3,2,0,0,0,0
3,4,0,0,0,0,0
4,5,4,0,0,0,0
5,6,2,0,0,0,0
6,7,1,0,0,0,0
7,8,0,0,0,0,0
8,9,0,0,0,0,0
9,10,0,0,0,0,0


In [11]:
# Release long-lived objects that hold DB/file handles
import gc

for obj_name in ["memory", "registry", "store"]:
    obj = globals().get(obj_name)
    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed: {obj_name}")
        except Exception as e:
            print(f"Failed to close {obj_name}: {e}")

for obj_name in ["memory", "registry", "store"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()
print("Released DB/file handles and cleared related globals.")

Closed: memory
Closed: registry
Released DB/file handles and cleared related globals.


## 6. Release DB handles (optional but recommended)

Run this cell after finishing Notebook 04 to release DuckDB file locks without restarting the kernel.

## ✅ Notebook 04 Complete

The dual-track closed loop has run. The system has:
- **Explorer track**: Generated LLM hypotheses guided by RAG context, synthesized expressions
- **Skeleton track**: Retrieved proven skeleton templates, generated field-substituted variants
- **TrackBandit**: Adaptively allocated simulation budget between both tracks
- **SkeletonExtractor**: Deposited qualified/soft-qualified alphas back into SkeletonRegistry
- Filtered near-duplicates (strict novelty for Explorer, field_coverage for Skeleton)
- Reflected on failures and stored insights for next round

**Next**: `05_results_and_novelty_analysis.ipynb` for full analysis with skeleton family trees.

## ✅ Notebook 04 Complete

The closed loop has run. The system has:
- Generated LLM hypotheses guided by RAG context
- Synthesized and locally validated expressions
- Filtered out near-duplicates using novelty scoring
- Submitted to WQB for backtesting
- Reflected on failures and stored insights for next round

**Next**: `05_results_and_novelty_analysis.ipynb` for full analysis.